# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Citation**: Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load as an mlcroissant dataset
dataset = mlc.Dataset(croissant_url)

# Explore the top-level metadata
print('Dataset name:', dataset.metadata.name)
print('Description:', dataset.metadata.description)
print('Version:', dataset.metadata.version)
print('Published:', dataset.metadata.datePublished)
print('License:', dataset.metadata.license)


## 2. Data Overview
Review available record sets, fields, and their `@id`s/reference keys.

We explore and print all record sets, including their `@id`s and the fields they contain using their `@id`s. This allows referencing by identifier in later steps.

In [ ]:
# Print record set @ids and their field @ids
print('=== RecordSets ===')

record_set_ids = []
for recset in dataset.metadata.recordSet:
    recset_id = recset['@id'] if isinstance(recset, dict) else recset
    record_set_ids.append(recset_id)

    recset_metadata = dataset.metadata.get_by_id(recset_id)

    print(f'RecordSet @id: {recset_id}')
    if hasattr(recset_metadata, 'field'):
        fields = recset_metadata.field
        print('  Fields:')
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            field_obj = dataset.metadata.get_by_id(field_id)
            print(f'    - {field_id} ({getattr(field_obj, "name", "")})')
    else:
        print('  (No field info available)')
    print('')

# Example: print one sample record from each set (if any)
for recset_id in record_set_ids:
    print(f'One record from {recset_id}:')
    try:
        for i, x in enumerate(dataset.records(record_set=recset_id)):
            print(json.dumps(x, indent=2))
            break
    except Exception as e:
        print(f'  Error reading records ({e})')
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We use the record set(s) and field `@id`s from the overview above.

In [ ]:
"""
You may find more than one record set in `record_set_ids`. For demonstration, we'll use the first one. 
"""

# Choose record sets to extract (use their @id)
record_sets = record_set_ids
dataframes = {}

for record_set_id in record_sets:
    try:
        # Read all records into DataFrame
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f'Loaded {len(df)} records for {record_set_id}. Columns: {df.columns.tolist()}')
    except Exception as e:
        print(f'Could not load data for {record_set_id}: {e}')

# For the main record set, show few rows
main_record_set = record_sets[0] if record_sets else None
if main_record_set and main_record_set in dataframes:
    print(f'\nColumns in {main_record_set}:\n', dataframes[main_record_set].columns.tolist())
    dataframes[main_record_set].head()
else:
    print('No records available for preview.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing, and grouping data for further exploration.


In [ ]:
from pandas.api.types import is_numeric_dtype

# Identify a numeric field from the DataFrame
df = dataframes.get(main_record_set, None)
numeric_field = None
if df is not None:
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field = col
            break

if df is None or numeric_field is None:
    print('Could not find numeric field for analysis.')
else:
    print(f'Using numeric field: {numeric_field}')

    threshold = df[numeric_field].quantile(0.75) if df[numeric_field].nunique() > 10 else df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a categorical field if one exists
    group_field = None
    for col in df.columns:
        if not is_numeric_dtype(df[col]) and df[col].nunique() < 10:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        print('No suitable categorical group field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution
if df is not None and numeric_field is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If there is a group field, show boxplot
    if group_field:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all key entities by their Croissant `@id`. You can adapt and extend these steps for deeper domain-specific analyses or machine learning preparation.

> **Key steps:**
> - Load the dataset and review metadata and structure using Croissant identifiers.
> - Extract, inspect, and filter tabular data using record set and field `@id`s.
> - Perform simple exploratory data analysis and visualize numeric distributions.